# NER propre à Djibouti — Baselines (Mako)

Master 2 IAMD, Université de Djibouti — module TALN.

**Question :** peut-on extraire les institutions, lieux, personnes et dates dans des textes français de Djibouti ?

**Plan du notebook**
1. Charger les données et les annotations
2. Baseline 1 : spaCy `fr_core_news_sm` seul
3. Améliorations, une par une :
   - **3a** règles `EntityRuler` (dates, ministères, comités, lexique local, titre + nom)
   - **3b** filtrage après spaCy, aligné sur le guide  ← *section en cours*
   - 3c comparaison `sm` / `md` / `lg`
   - 3d score souple (frontières) en plus du score strict
4. Tableau récapitulatif (test)
5. Analyse de 10 erreurs
6. Démo avec `displacy`

## Section 1 — Charger les données

**Ce qu'on fait :**
- on lit les deux fichiers : les textes (`corpus_ner.csv`) et les annotations (`annotations.csv`) ;
- pour chaque segment, on regroupe ses entités sous forme de triplets `(debut, fin, type)` ;
- on sépare le **dev** (pour écrire les règles) et le **test** (pour mesurer, sans y toucher).

**Pourquoi des triplets ?** C'est le format attendu par `src/evaluate.py`. Exemple : dans `"Il est à Djibouti."`, l'entité `Djibouti` donne le triplet `(10, 18, "LOC")`.

In [1]:
from pathlib import Path
from collections import Counter

import pandas as pd

# Le notebook est dans notebooks/ : les données sont un dossier plus haut.
RACINE = Path("..").resolve()
DATA = RACINE / "data"

corpus = pd.read_csv(DATA / "corpus_ner.csv", encoding="utf-8")
annotations = pd.read_csv(DATA / "annotations.csv", encoding="utf-8", keep_default_na=False)

print("Segments dans le corpus :", len(corpus))
print("Lignes d'annotation     :", len(annotations))
print("Colonnes du corpus      :", list(corpus.columns))
print("Colonnes des annotations:", list(annotations.columns))

Segments dans le corpus : 126
Lignes d'annotation     : 84
Colonnes du corpus      : ['id', 'titre', 'rubrique', 'url', 'date', 'fr', 'split']
Colonnes des annotations: ['id_segment', 'split', 'texte', 'entite', 'type', 'debut', 'fin', 'a_verifier', 'remarque']


**Attention : segments sans entité.** Le fichier d'annotations n'a aucune ligne pour un segment sans entité. Mais ces segments comptent dans l'évaluation : si le modèle y invente une entité, c'est une erreur (fausse). On leur donne donc une liste **vide**.

In [2]:
# gold[id_segment] = liste de triplets (debut, fin, type)
gold = {id_seg: [] for id_seg in corpus["id"]}

for ligne in annotations.itertuples():
    gold[ligne.id_segment].append((int(ligne.debut), int(ligne.fin), ligne.type))

# On trie pour avoir un ordre stable
for id_seg in gold:
    gold[id_seg].sort()

# Vérification : tous les segments annotés existent dans le corpus ?
inconnus = set(annotations["id_segment"]) - set(corpus["id"])
print("Segments annotés absents du corpus :", len(inconnus))

n_vides = sum(1 for ents in gold.values() if not ents)
print("Segments avec au moins une entité  :", len(gold) - n_vides)
print("Segments sans entité (liste vide)  :", n_vides)

Segments annotés absents du corpus : 0
Segments avec au moins une entité  : 48
Segments sans entité (liste vide)  : 78


**Dev et test.** On crée, pour chaque split, trois listes de même longueur et dans le même ordre : les ids, les textes, et les entités de la vérité terrain.

In [3]:
def charger_split(nom):
    sous = corpus[corpus["split"] == nom]
    ids = list(sous["id"])
    textes = list(sous["fr"])
    verite = [gold[i] for i in ids]
    return ids, textes, verite

ids_dev, textes_dev, gold_dev = charger_split("dev")
ids_test, textes_test, gold_test = charger_split("test")

print("dev  :", len(ids_dev), "segments")
print("test :", len(ids_test), "segments")

dev  : 84 segments
test : 42 segments


**Comptes réels par split et par type.**

In [4]:
TYPES = ["ORG", "LOC", "PER", "DATE", "FORMATION"]

def compter(gold_liste):
    c = Counter(t for ents in gold_liste for (_, _, t) in ents)
    return {t: c.get(t, 0) for t in TYPES}

tableau = pd.DataFrame({"dev": compter(gold_dev), "test": compter(gold_test)})
tableau["total"] = tableau["dev"] + tableau["test"]
tableau.loc["TOTAL"] = tableau.sum()
tableau

,dev,test,total
ORG,17,13,30
LOC,16,4,20
PER,14,7,21
DATE,10,3,13
FORMATION,0,0,0
TOTAL,57,27,84


**Lecture :**
- le **test est petit** : peu d'entités par type, donc un seul changement fait bouger le score de plusieurs points ;
- **FORMATION = 0** : aucun diplôme dans ce corpus, on n'en a pas inventé.

In [5]:
# Un exemple concret : un segment, ses triplets, et le texte coupé avec ces positions
exemple = "A03_S002"
texte = corpus.loc[corpus["id"] == exemple, "fr"].iloc[0]
print(texte, "\n")
for debut, fin, type_ in gold[exemple]:
    print(f"({debut}, {fin}, {type_!r:6})  ->  {texte[debut:fin]!r}")

Réunis en Assemblée générale extraordinaire au siège de la Fédération, ils ont voté à l'unanimité la révocation du Comité directeur et de son président, Souleiman Hassan Waberi. 

(115, 131, 'ORG' )  ->  'Comité directeur'
(153, 176, 'PER' )  ->  'Souleiman Hassan Waberi'


In [6]:
# Contrôle final : pour TOUTES les annotations, le texte coupé doit être l'entité notée
ok = all(
    corpus.loc[corpus["id"] == l.id_segment, "fr"].iloc[0][l.debut:l.fin] == l.entite
    for l in annotations.itertuples()
)
print("Toutes les positions correspondent au texte :", ok)

Toutes les positions correspondent au texte : True


**Fin de la Section 1.** On a `gold_dev`, `gold_test`, `textes_dev`, `textes_test`, `ids_dev`, `ids_test`. La Section 2 les utilisera pour la baseline spaCy.

## Section 2 — Baseline 1 : spaCy seul

**Ce qu'on fait :** on passe nos textes dans le modèle `fr_core_news_sm` **sans aucune règle ajoutée**, puis on compare ce qu'il trouve avec notre vérité terrain.

**Le modèle, en bref :** c'est un petit modèle français de spaCy, déjà entraîné par ses auteurs. Il ne connaît pas Djibouti en particulier. Cellule suivante : sa fiche d'identité (nom, version, langue, étiquettes).

In [7]:
import sys
import spacy

nlp = spacy.load("fr_core_news_sm")
meta = nlp.meta

print("spaCy (bibliothèque) :", spacy.__version__)
print("Modèle               :", meta["lang"] + "_" + meta["name"])
print("Version du modèle    :", meta["version"])
print("Langue               :", meta["lang"])
print("Composants           :", nlp.pipe_names)
print("Étiquettes du NER    :", nlp.get_pipe("ner").labels)
print("Données d'entraînement:", [s["name"] for s in meta["sources"]])

spaCy (bibliothèque) : 3.8.18
Modèle               : fr_core_news_sm
Version du modèle    : 3.8.0
Langue               : fr
Composants           : ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Étiquettes du NER    : ('LOC', 'MISC', 'ORG', 'PER')
Données d'entraînement: ['UD French Sequoia v2.8', 'WikiNER', 'spaCy lookups data']


**À retenir (limites) :**
- le modèle ne prédit que **4 étiquettes** : `PER`, `ORG`, `LOC`, `MISC`. Il ne connaît ni `DATE` ni `FORMATION` ;
- c'est le plus petit modèle (`sm`) : rapide mais moins précis ;
- il a été entraîné sur des textes généraux (articles Wikipédia, corpus Sequoia), pas sur la presse de Djibouti.

**On importe `src/evaluate.py`** (le calcul précision / rappel / F1) :

In [8]:
sys.path.insert(0, str(RACINE / "src"))
from evaluate import evaluer, afficher, erreurs

print("Fonctions importées :", evaluer.__name__, afficher.__name__, erreurs.__name__)

Fonctions importées : evaluer afficher erreurs


**Prédire.** La fonction `predire` fait trois choses :
1. elle passe chaque texte dans spaCy ;
2. elle lit pour chaque entité `start_char` et `end_char` : ce sont des positions **en caractères**, comme dans nos annotations ;
3. elle garde `PER`, `ORG`, `LOC` et **ignore `MISC`**, car nous n'avons pas ce type dans notre guide (MISC veut dire « divers » : spaCy y met tout ce qui n'entre pas ailleurs).

Elle compte aussi ce qu'elle ignore, pour qu'on voie ce qu'on perd.

In [9]:
GARDES = ["PER", "ORG", "LOC"]

def predire(textes):
    """Retourne (prédictions, compteur des étiquettes ignorées).
    prédictions = une liste par texte de triplets (debut, fin, type)."""
    resultat, ignorees = [], Counter()
    for doc in nlp.pipe(textes):
        ents = []
        for ent in doc.ents:
            if ent.label_ in GARDES:
                ents.append((ent.start_char, ent.end_char, ent.label_))
            else:
                ignorees[ent.label_] += 1
        resultat.append(ents)
    return resultat, ignorees

pred_dev, ignorees_dev = predire(textes_dev)
pred_test, ignorees_test = predire(textes_test)

def total(pred):
    return Counter(t for ents in pred for (_, _, t) in ents)

print("TEST — entités gardées     :", dict(total(pred_test)), "| ignorées :", dict(ignorees_test))
print("DEV  — entités gardées     :", dict(total(pred_dev)),  "| ignorées :", dict(ignorees_dev))

TEST — entités gardées     : {'ORG': 19, 'LOC': 7, 'PER': 4} | ignorées : {'MISC': 2}
DEV  — entités gardées     : {'ORG': 19, 'PER': 21, 'LOC': 29} | ignorées : {'MISC': 4}


**Vérité terrain filtrée.** Le score principal porte sur `PER`, `ORG`, `LOC` seulement, car ce sont les seuls types que spaCy peut prédire. On retire donc `DATE` (et `FORMATION`) de la vérité terrain **pour ce calcul**. Sinon on punirait spaCy pour des dates qu'il ne sait pas prédire.

In [10]:
def filtrer(gold_liste, types):
    return [[e for e in ents if e[2] in types] for ents in gold_liste]

gold_test_3 = filtrer(gold_test, GARDES)
gold_dev_3 = filtrer(gold_dev, GARDES)

print("Entités de la vérité terrain gardées (PER, ORG, LOC) :")
print("  test :", sum(len(e) for e in gold_test_3), "sur", sum(len(e) for e in gold_test))
print("  dev  :", sum(len(e) for e in gold_dev_3), "sur", sum(len(e) for e in gold_dev))

Entités de la vérité terrain gardées (PER, ORG, LOC) :
  test : 24 sur 27
  dev  : 47 sur 57


### Résultat sur le TEST (score principal)

Une entité est **juste** seulement si début, fin **et** type sont exacts. `TOUT` = PER + ORG + LOC.

In [11]:
res_test = evaluer(gold_test_3, pred_test)
afficher(res_test)

Type         Précision    Rappel      F1  Support
LOC               0.29      0.50    0.36        4
ORG               0.47      0.69    0.56       13
PER               0.75      0.43    0.55        7
TOUT              0.47      0.58    0.52       24


**Comment lire :**
- **Précision** : parmi ce que spaCy a trouvé, quelle part est juste ;
- **Rappel** : parmi ce qu'il fallait trouver, quelle part spaCy a trouvée ;
- **Support** : nombre d'entités réelles du type dans le test ;
- le test est **petit** : un seul changement fait bouger un score de plusieurs points.

### DATE : non prédit par spaCy

spaCy `fr_core_news_sm` n'a pas d'étiquette `DATE`. Il ne peut donc **jamais** en prédire. On le montre pour le rapport, mais on ne le mélange pas au score principal.

In [12]:
def date_seule(gold_liste):
    return filtrer(gold_liste, ["DATE"])

for nom, gold_x in [("test", gold_test), ("dev", gold_dev)]:
    gd = date_seule(gold_x)
    vide = [[] for _ in gd]                     # spaCy ne prédit aucune DATE
    r = evaluer(gd, vide)["DATE"]
    print(f"DATE ({nom}) : support = {r[3]}, prédites par spaCy = 0, rappel = {r[1]:.2f}  -> non prédit par spaCy")
    print("  (précision non définie : 0 prédiction)")

DATE (test) : support = 3, prédites par spaCy = 0, rappel = 0.00  -> non prédit par spaCy
  (précision non définie : 0 prédiction)
DATE (dev) : support = 10, prédites par spaCy = 0, rappel = 0.00  -> non prédit par spaCy
  (précision non définie : 0 prédiction)


### Mêmes chiffres sur le DEV (à titre indicatif)

On ne s'en sert **pas** pour décider quoi que ce soit : le dev sert plus tard à écrire les règles (Section 3). Il est seulement là pour voir si les chiffres du test sont d'un ordre de grandeur comparable.

In [13]:
res_dev = evaluer(gold_dev_3, pred_dev)
afficher(res_dev)

Type         Précision    Rappel      F1  Support
LOC               0.41      0.75    0.53       16
ORG               0.58      0.65    0.61       17
PER               0.52      0.79    0.63       14
TOUT              0.49      0.72    0.59       47


### Combien d'entités MISC sont ignorées ?

Ce sont des entités que spaCy a trouvées mais qu'on ne compare pas (pas de type `MISC` dans notre guide). Voici lesquelles, pour voir si on perd quelque chose d'important.

In [14]:
print("MISC ignorées : test =", ignorees_test.get("MISC", 0), "| dev =", ignorees_dev.get("MISC", 0))
print("Autres étiquettes ignorées :", {k: v for k, v in (ignorees_test + ignorees_dev).items() if k != "MISC"} or "aucune")

print("\nExemples de MISC (test) :")
for doc, id_ in zip(nlp.pipe(textes_test), ids_test):
    for ent in doc.ents:
        if ent.label_ == "MISC":
            print(f"  {id_}  {ent.text!r}")

MISC ignorées : test = 2 | dev = 4
Autres étiquettes ignorées : aucune

Exemples de MISC (test) :


  A03_S007  'Me Abeyazide'


  A11_S001  'Formation professionnelle'


### À quoi ressemblent les erreurs ? 5 segments du test

**Choix des segments :** les 5 **premiers** segments du test (dans l'ordre du fichier) qui contiennent au moins une erreur sur `PER/ORG/LOC`. Ce n'est donc **pas** un échantillon représentatif : on montre exprès des segments avec erreurs.

- *manquée* = dans la vérité terrain, mais spaCy ne l'a pas trouvée avec les mêmes frontières ET le même type ;
- *fausse* = spaCy l'a prédite, mais elle n'est pas dans la vérité terrain.

In [15]:
def montrer(id_, texte, g, p):
    print(f"[{id_}] {texte}")
    print("  spaCy  :", [(texte[d:f], t) for d, f, t in p] or "aucune")
    print("  vérité :", [(texte[d:f], t) for d, f, t in g] or "aucune")
    for genre, ent, mot in erreurs(texte, g, p):
        print(f"  -> {genre:8} {mot!r} ({ent[2]})")
    print()

compte = 0
for id_, texte, g, p in zip(ids_test, textes_test, gold_test_3, pred_test):
    if erreurs(texte, g, p):
        montrer(id_, texte, g, p)
        compte += 1
        if compte == 5:
            break

[A03_S002] Réunis en Assemblée générale extraordinaire au siège de la Fédération, ils ont voté à l'unanimité la révocation du Comité directeur et de son président, Souleiman Hassan Waberi.
  spaCy  : [('Réunis', 'LOC'), ('Assemblée générale', 'ORG'), ('Fédération', 'LOC'), ('Comité directeur', 'ORG'), ('Souleiman Hassan Waberi', 'PER')]
  vérité : [('Comité directeur', 'ORG'), ('Souleiman Hassan Waberi', 'PER')]
  -> fausse   'Réunis' (LOC)
  -> fausse   'Assemblée générale' (ORG)
  -> fausse   'Fédération' (LOC)

[A03_S004] L'Assemblée générale extraordinaire avait été convoquée à la demande des clubs membres afin d'examiner la situation de la Fédération et son avenir institutionnel.
  spaCy  : [('Assemblée générale', 'ORG'), ('Fédération', 'ORG')]
  vérité : aucune
  -> fausse   'Assemblée générale' (ORG)
  -> fausse   'Fédération' (ORG)

[A03_S007] La réunion s'est tenue dans un cadre formel, sous le contrôle du consultant Me Abeyazide et de l'huissier de justice Habib Alwan.
  spaC

**Fin de la Section 2.** On a `pred_test`, `pred_dev` et les scores de la baseline 1 (`res_test`, `res_dev`). La Section 3 ajoutera des règles locales (`EntityRuler`) en regardant **seulement le dev**.

## Section 3 — Améliorer la baseline, étape par étape

### Protocole (à respecter à chaque étape)
1. On écrit et on règle les règles en regardant **le DEV seulement**. Aucune règle ne vient du TEST.
2. Quand une version est **figée**, on la mesure sur le TEST **une seule fois**, et on ajoute une ligne au tableau récapitulatif.
3. Les chiffres du DEV sont **optimistes** : les règles ont été écrites en le regardant. Ils servent à voir l'effet d'une règle, pas à annoncer un score.
4. Si on corrige une règle après avoir vu des erreurs du test, cette version est enregistrée **à part**, étiquetée « après analyse d'erreurs du test (optimiste) ».

**Une précision d'honnêteté.** J'ai lu tous les segments (dev ET test) pour les annoter, avant de fixer ce protocole. Je ne peux donc pas dire que le test m'est inconnu. Pour limiter le risque : les règles viennent soit du dev, soit de connaissances générales (jours, mois, régions…), soit de la consigne. La provenance de chaque groupe est affichée ci-dessous, et un contrôle montre quelles entrées du lexique apparaissent dans le texte du test.

---
### 3a — `EntityRuler` : règles générales + lexique local

**Idée.** Le `EntityRuler` est un composant spaCy qui marque des entités avec des **règles écrites à la main**. On le place **avant** le NER (`before="ner"`) : les entités des règles sont posées d'abord, et le modèle statistique travaille autour d'elles.

Exemple : la règle « jour + nombre + mois » marque `lundi 5 octobre` comme DATE, sans que le modèle ait besoin de la connaître.

Les règles sont dans `src/regles_locales.py`, rangées en **groupes** pour mesurer l'effet de chacun.

In [16]:
import importlib
import regles_locales as R
importlib.reload(R)

lignes = []
for nom, regles in R.GROUPES.items():
    n = "composant à part" if regles == "composant" else len(regles)
    lignes.append({"groupe": nom, "nb de règles": n, "provenance": R.SOURCES[nom]})
pd.set_option("display.max_colwidth", 140)
pd.DataFrame(lignes)

,groupe,nb de règles,provenance
0,dates,6,"dates : mois et jours = connaissance du français ; formes (5 octobre, 6, 7 et 8 octobre, 2026-2027) vues dans le DEV"
1,org_ministeres,8,« ministère de… » / « Secrétariat d'État chargé… » : demandé dans la consigne (3a) ; adjectifs = connaissance générale
2,org_organes,4,« Comité … » / « Conseil régional » : demandé dans la consigne (3a) / règles du guide
3,org_entreprises,1,"« X Y Industries » : forme vue dans le DEV (A02), écrite de façon générale"
4,lexique_org,23,mélange : entrées vues dans le DEV + institutions connues a priori (voir LEXIQUE_ORG)
5,lexique_loc,28,"mélange : entrées vues dans le DEV + pays, régions, villes connues a priori (voir LEXIQUE_LOC)"
6,per_titres,composant à part,"titre + nom : consigne (3a) ; formes vues dans le DEV (« Secrétaire générale, Louise… »)"


**Les règles, une par une, sur des phrases inventées** (pas des phrases du corpus). Chaque phrase montre une règle.

In [17]:
nlp_demo = R.construire(groupes=list(R.GROUPES))

essais = [
    ("dates : jour + date",           "La réunion aura lieu mardi 3 mars à Obock."),
    ("dates : plage avec « et »",     "Les travaux durent les 12 et 13 juin 2027."),
    ("dates : liste",                 "Le stage est prévu les 4, 5 et 6 mai."),
    ("dates : jour + moment",         "Le concert a eu lieu samedi soir."),
    ("dates : année scolaire",        "Pour l'année 2027-2028, de nouvelles classes ouvrent."),
    ("ORG : ministère",               "Elle travaille au ministère de la Santé et de la Population."),
    ("ORG : secrétariat d'État",      "Le Secrétariat d'État chargé des Transports a publié un rapport."),
    ("ORG : comité",                  "Le Comité technique s'est réuni."),
    ("ORG : conseil régional",        "Le Conseil régional d'Obock a voté le budget."),
    ("ORG : entreprise « Industries »", "La visite s'est faite à l'usine Sahara Industries."),
    ("ORG/LOC : lexique",             "L'UNICEF et l'Université de Djibouti coopèrent avec Dikhil."),
    ("PER : titre + nom (fonction)",  "Le ministre Ahmed Ali Youssouf a parlé."),
    ("PER : titre + nom (virgule)",   "Le directeur général, Samira Omar Hassan, a salué l'initiative."),
    ("PER : civilité",                "Le Dr Yacin Moussa a ouvert la séance."),
]
for nom, phrase in essais:
    doc = nlp_demo(phrase)
    print(f"{nom:34} -> {[(e.text, e.label_) for e in doc.ents]}")

dates : jour + date                -> [('mardi 3 mars', 'DATE'), ('Obock', 'LOC')]
dates : plage avec « et »          -> [('12 et 13 juin 2027', 'DATE')]
dates : liste                      -> [('4, 5 et 6 mai', 'DATE')]
dates : jour + moment              -> [('samedi soir', 'DATE')]
dates : année scolaire             -> [('2027-2028', 'DATE')]
ORG : ministère                    -> [('ministère de la Santé et de la Population', 'ORG')]
ORG : secrétariat d'État           -> [("Secrétariat d'État chargé des Transports", 'ORG')]
ORG : comité                       -> [('Comité technique', 'ORG')]
ORG : conseil régional             -> [("Conseil régional d'Obock", 'ORG')]


ORG : entreprise « Industries »    -> [('Sahara Industries', 'ORG')]
ORG/LOC : lexique                  -> [('UNICEF', 'ORG'), ('Université de Djibouti', 'ORG'), ('Dikhil', 'LOC')]
PER : titre + nom (fonction)       -> [('Ahmed Ali Youssouf', 'PER')]
PER : titre + nom (virgule)        -> [('Samira Omar Hassan', 'PER')]
PER : civilité                     -> [('Yacin Moussa', 'PER')]


**Lecture :** pour **PER**, le titre (« Dr », « ministre ») reste **hors** de l'entité, comme dans notre guide. Seul le nom est marqué.

**Effet de chaque groupe, mesuré sur le DEV seulement.** On part de la baseline et on ajoute **un seul groupe à la fois**. Rappel : le DEV est optimiste (règles écrites en le regardant).

In [18]:
TYPES_PRED = ["PER", "ORG", "LOC", "DATE"]

def predire_types(nlp_x, textes, types=TYPES_PRED):
    """Comme predire(), mais pour n'importe quel pipeline et pour les types demandés."""
    return [[(e.start_char, e.end_char, e.label_) for e in doc.ents if e.label_ in types]
            for doc in nlp_x.pipe(textes)]

def scores(gold_liste, pred_liste):
    """(résultats PER/ORG/LOC avec TOUT, résultat DATE ou None)"""
    r3 = evaluer(filtrer(gold_liste, GARDES), filtrer(pred_liste, GARDES))
    gd = filtrer(gold_liste, ["DATE"])
    rd = evaluer(gd, filtrer(pred_liste, ["DATE"])).get("DATE")
    return r3, rd

ablation = []
for groupes in [[]] + [[g] for g in R.GROUPES] + [list(R.GROUPES)]:
    r3, rd = scores(gold_dev, predire_types(R.construire(groupes=groupes), textes_dev))
    nom = "baseline (aucun groupe)" if not groupes else ("TOUS les groupes" if len(groupes) > 1 else "+ " + groupes[0])
    ablation.append({"version (DEV)": nom,
                     "TOUT P": r3["TOUT"][0], "TOUT R": r3["TOUT"][1], "TOUT F1": r3["TOUT"][2],
                     "DATE F1": rd[2] if rd else 0.0})
pd.DataFrame(ablation).round(2)

,version (DEV),TOUT P,TOUT R,TOUT F1,DATE F1
0,baseline (aucun groupe),0.49,0.72,0.59,0.0
1,+ dates,0.49,0.72,0.59,1.0
2,+ org_ministeres,0.49,0.72,0.59,0.0
3,+ org_organes,0.49,0.72,0.59,0.0
4,+ org_entreprises,0.54,0.79,0.64,0.0
5,+ lexique_org,0.52,0.79,0.63,0.0
6,+ lexique_loc,0.55,0.81,0.66,0.0
7,+ per_titres,0.51,0.74,0.61,0.0
8,TOUS les groupes,0.64,0.96,0.77,1.0


**Lecture :**
- `dates` ne change pas le score PER/ORG/LOC, mais fait passer **DATE** de 0 à une valeur élevée (spaCy n'en prédisait aucune) ;
- les groupes `org_ministeres` et `org_organes` n'ont **aucun effet sur le dev** : spaCy trouvait déjà ces cas, ou ils n'y sont pas. On les garde parce que la consigne les demande et qu'ils sont généraux, mais on **ne peut pas** dire qu'ils aident, tant qu'on n'a pas vu le test ;
- le lexique des lieux et `org_entreprises` sont ceux qui aident le plus sur le dev : c'est normal, une partie de leurs entrées vient du dev.

### Contrôle de transparence avant de figer
On liste les entrées du lexique qui apparaissent dans le texte du **test**. Ce n'est qu'un **diagnostic** : on ne change aucune règle après l'avoir regardé.

In [19]:
textes_test_complet = " ".join(textes_test)
entrees = [("lexique_org", m) for m in R.LEXIQUE_ORG] + [("lexique_loc", m) for m in R.LEXIQUE_LOC]
presentes = [(g, m) for g, m in entrees if m in textes_test_complet]
print(f"Entrées du lexique : {len(entrees)} | présentes dans le texte du TEST : {len(presentes)}")
for g, m in presentes:
    vu_dev = any(m in t for t in textes_dev)
    print(f"  {g:12} {m!r:30} vue aussi dans le dev : {'oui' if vu_dev else 'NON (connaissance a priori)'}")

Entrées du lexique : 51 | présentes dans le texte du TEST : 1
  lexique_loc  'Ali-Sabieh'                   vue aussi dans le dev : NON (connaissance a priori)


### Mesure unique sur le TEST (version « 3a »)

La version 3a est **figée** : tous les groupes de règles ci-dessus. On la mesure sur le test **une seule fois**. Score principal = `PER`, `ORG`, `LOC`. `DATE` est à part.

In [20]:
nlp_3a = R.construire(groupes=list(R.GROUPES))
pred_test_3a = predire_types(nlp_3a, textes_test)
res_test_3a, res_date_3a = scores(gold_test, pred_test_3a)

print("Version 3a — TEST (PER, ORG, LOC)")
afficher(res_test_3a)
print()
print("DATE (à part) : précision = %.2f, rappel = %.2f, F1 = %.2f, support = %d" % res_date_3a)

Version 3a — TEST (PER, ORG, LOC)
Type         Précision    Rappel      F1  Support
LOC               0.44      1.00    0.62        4
ORG               0.72      1.00    0.84       13
PER               0.83      0.71    0.77        7
TOUT              0.67      0.92    0.77       24

DATE (à part) : précision = 1.00, rappel = 1.00, F1 = 1.00, support = 3


### Tableau récapitulatif (TEST)

Une ligne par version et par type. `DATE` est à part : spaCy seul n'en prédit pas, donc sa précision n'est **pas définie** (0 prédiction) pour la baseline 1.

In [21]:
recap = {}   # nom de version -> (résultats PER/ORG/LOC, résultat DATE)

def enregistrer(nom, res3, res_date):
    recap[nom] = (res3, res_date)

def tableau_recap():
    lignes = []
    for nom, (res3, res_date) in recap.items():
        for type_ in ["PER", "ORG", "LOC", "TOUT"]:
            p, r, f, s = res3[type_]
            lignes.append({"version": nom, "type": type_ if type_ != "TOUT" else "TOUT (PER+ORG+LOC)",
                           "précision": round(p, 2), "rappel": round(r, 2), "F1": round(f, 2), "support": s})
        if res_date is not None:
            p, r, f, s = res_date
            lignes.append({"version": nom, "type": "DATE (à part)", "précision": round(p, 2),
                           "rappel": round(r, 2), "F1": round(f, 2), "support": s})
    return pd.DataFrame(lignes)

# Baseline 1 (Section 2) : DATE non prédit -> précision non définie, affichée 0.00
date_baseline = evaluer(filtrer(gold_test, ["DATE"]), [[] for _ in gold_test])["DATE"]
enregistrer("Baseline 1 : spaCy seul", res_test, date_baseline)
enregistrer("3a : + règles EntityRuler", res_test_3a, res_date_3a)
tableau_recap()

,version,type,précision,rappel,F1,support
0,Baseline 1 : spaCy seul,PER,0.75,0.43,0.55,7
1,Baseline 1 : spaCy seul,ORG,0.47,0.69,0.56,13
2,Baseline 1 : spaCy seul,LOC,0.29,0.50,0.36,4
3,Baseline 1 : spaCy seul,TOUT (PER+ORG+LOC),0.47,0.58,0.52,24
4,Baseline 1 : spaCy seul,DATE (à part),0.00,0.00,0.00,3
5,3a : + règles EntityRuler,PER,0.83,0.71,0.77,7
6,3a : + règles EntityRuler,ORG,0.72,1.00,0.84,13
7,3a : + règles EntityRuler,LOC,0.44,1.00,0.62,4
8,3a : + règles EntityRuler,TOUT (PER+ORG+LOC),0.67,0.92,0.77,24
9,3a : + règles EntityRuler,DATE (à part),1.00,1.00,1.00,3


**Rappels pour lire ce tableau :**
- le test est **petit** (24 entités PER/ORG/LOC, 3 DATE) : un seul changement déplace un score de plusieurs points ;
- la ligne **DATE** de la baseline 1 : précision « non définie » (aucune prédiction), affichée 0.00 par le script ;
- les entrées de lexique « vues dans le dev » ont été choisies en regardant le dev ; les autres viennent de connaissances générales (voir le contrôle plus haut).

**⚠ Limite importante de la version 3a (à dire au jury).**
- Les groupes `org_ministeres` et `org_organes` n'ont **aucun effet sur le dev** (tableau d'ablation). On ne peut donc pas les avoir validés sur le dev.
- Ces règles (« ministère de… », « Secrétariat d'État chargé… », « Comité … ») viennent de la **consigne**, écrite après que j'ai annoté tout le corpus, test compris. Elles ressemblent à des cas qui existent dans le test.
- Le score 3a sur le test est donc **probablement optimiste** : il ne prouve pas que les règles généraliseront à de nouveaux textes.
- Le LOC du test contient 4 entités, dont 2 « Ali-Sabieh » (qui est dans le lexique, voir le contrôle plus haut) : un rappel LOC de 1.00 ne veut pas dire grand-chose sur 4 entités.
- Pour avoir une mesure propre, il faudrait annoter quelques segments **nouveaux** (articles non utilisés) comme second test.

**Fin de l'étape 3a.** Prochaine étape (3b) : filtrer après spaCy, selon le guide (mentions courtes, événements, titres).

### 3b — Filtrer après spaCy, selon le guide

**Idée.** Après le NER, on **enlève ou corrige** des entités qui ne respectent pas notre guide. Ce sont des erreurs qu'on repère sur le **dev**. Les règles de 3a sont inchangées.

**Les 7 filtres** (dans `src/regles_locales.py`, composant `filtre_guide`) :

| Filtre | Ce qu'il fait | Règle du guide |
|---|---|---|
| `minuscule` | enlève une entité qui commence par une minuscule (sauf « ministère ») | un nom propre commence par une majuscule |
| `fonction` | enlève une ORG/LOC qui commence par une fonction (« Secrétaire générale de l'… ») | fonction = pas une entité |
| `mention_courte` | enlève « Organisation », « Fédération », « Comité »… seuls | mentions courtes non annotées |
| `evenement` | enlève les entités dans un nom d'événement (« Challenge… », « Mission économique… ») | événements hors types |
| `titre_per` | retire « Me », « Dr », « inspecteur général »… au début d'un PER | titre hors du PER |
| `debut_phrase` | enlève un mot unique en début de phrase qui n'est pas un nom propre (« Détecter ») | |
| `domaines_ministre` | enlève les « lieux » dans « ministre de l'Agriculture, de l'Eau, de la Pêche… » | fonction = pas une entité |

**Un défaut trouvé sur le dev pendant l'écriture :** le filtre `minuscule` enlevait aussi « ministère de la Jeunesse et de la Culture » (qui commence par une minuscule et est bien une ORG). Le rappel baissait de 0.96 à 0.91. J'ai ajouté l'exception « ministère ».

**Un exemple réel du DEV pour chaque filtre** (avant / après, le filtre seul ajouté à la version 3a) :

In [22]:
STATUT = "test 1, utilisé pendant le développement (optimiste)"

nlp_3a_dev = R.construire(groupes=list(R.GROUPES))
docs_avant = [{(e.start_char, e.end_char, e.label_) for e in d.ents if e.label_ in TYPES_PRED}
              for d in nlp_3a_dev.pipe(textes_dev)]

for nom_filtre in R.FILTRES:
    nlp_f = R.construire(groupes=list(R.GROUPES), filtres=[nom_filtre])
    trouve = False
    for id_, texte, avant, doc in zip(ids_dev, textes_dev, docs_avant, nlp_f.pipe(textes_dev)):
        apres = {(e.start_char, e.end_char, e.label_) for e in doc.ents if e.label_ in TYPES_PRED}
        if apres != avant:
            retire = sorted(avant - apres); ajoute = sorted(apres - avant)
            print(f"[{nom_filtre}] {id_}")
            print("   enlevé :", [(texte[d:f], t) for d, f, t in retire])
            if ajoute:
                print("   ajouté :", [(texte[d:f], t) for d, f, t in ajoute])
            trouve = True
            break
    if not trouve:
        print(f"[{nom_filtre}] aucun effet sur le dev (règle issue du guide, non validée par le dev)")

[minuscule] A01_S004
   enlevé : [('aéroport international de', 'LOC')]


[fonction] A01_S001
   enlevé : [("Secrétaire générale de l'", 'ORG')]


[mention_courte] A08_S025
   enlevé : [('Organisation', 'ORG')]


[evenement] A01_S008
   enlevé : [('Mission économique de la Francophonie', 'ORG'), ("Afrique de l'Est", 'LOC')]


[titre_per] aucun effet sur le dev (règle issue du guide, non validée par le dev)


[debut_phrase] A08_S067
   enlevé : [('Détecter', 'PER')]


[domaines_ministre] A02_S001
   enlevé : [('Eau', 'LOC'), ('Pêche', 'LOC'), ('Élevage', 'LOC')]


**Effet de chaque filtre sur le DEV** (un filtre seul ajouté à 3a, puis tous). Le dev est optimiste : les filtres ont été choisis en regardant ses erreurs.

In [23]:
ablation_b = []
for filtres in [None] + [[f] for f in R.FILTRES] + [list(R.FILTRES)]:
    r3, rd = scores(gold_dev, predire_types(R.construire(groupes=list(R.GROUPES), filtres=filtres), textes_dev))
    nom = "3a (aucun filtre)" if filtres is None else ("TOUS les filtres" if len(filtres) > 1 else "+ " + filtres[0])
    ablation_b.append({"version (DEV)": nom, "TOUT P": r3["TOUT"][0], "TOUT R": r3["TOUT"][1],
                       "TOUT F1": r3["TOUT"][2], "DATE F1": rd[2] if rd else 0.0})
pd.DataFrame(ablation_b).round(2)

,version (DEV),TOUT P,TOUT R,TOUT F1,DATE F1
0,3a (aucun filtre),0.64,0.96,0.77,1.0
1,+ minuscule,0.68,0.96,0.80,1.0
2,+ fonction,0.69,0.96,0.80,1.0
3,+ mention_courte,0.65,0.96,0.78,1.0
4,+ evenement,0.71,0.96,0.82,1.0
5,+ titre_per,0.64,0.96,0.77,1.0
6,+ debut_phrase,0.65,0.96,0.78,1.0
7,+ domaines_ministre,0.70,0.96,0.81,1.0
8,TOUS les filtres,0.94,0.96,0.95,1.0


**Lecture :**
- les filtres augmentent surtout la **précision** (on enlève des fausses entités) ; le rappel ne doit pas baisser ;
- `titre_per` n'a **aucun effet sur le dev** : c'est une règle du guide, mais aucun cas du dev ne la déclenche. On ne peut pas dire qu'elle aide ;
- ces chiffres viennent du **dev**, sur lequel les filtres ont été réglés : ils sont optimistes.

### Mesure unique sur le TEST 1 (version « 3b »)

La version 3b est **figée** : règles 3a + les 7 filtres. Mesurée **une seule fois** sur le test 1.

In [24]:
nlp_3b = R.construire(groupes=list(R.GROUPES), filtres=list(R.FILTRES))
pred_test_3b = predire_types(nlp_3b, textes_test)
res_test_3b, res_date_3b = scores(gold_test, pred_test_3b)

print("Version 3b — TEST 1 (PER, ORG, LOC)")
afficher(res_test_3b)
print()
print("DATE (à part) : précision = %.2f, rappel = %.2f, F1 = %.2f, support = %d" % res_date_3b)

Version 3b — TEST 1 (PER, ORG, LOC)
Type         Précision    Rappel      F1  Support
LOC               0.80      1.00    0.89        4
ORG               0.93      1.00    0.96       13
PER               0.83      0.71    0.77        7
TOUT              0.88      0.92    0.90       24

DATE (à part) : précision = 1.00, rappel = 1.00, F1 = 1.00, support = 3


### Tableau récapitulatif (TEST 1)

Toutes les lignes portent la même étiquette : **test 1, utilisé pendant le développement (optimiste)**. Cela vaut aussi pour la baseline 1 et la version 3a : le test 1 a été lu pendant l'annotation, et la consigne des règles a été écrite après.

In [25]:
# On met la même étiquette sur les lignes déjà enregistrées (baseline 1 et 3a)
recap = {(k if STATUT in k else f"{k} [{STATUT}]"): v for k, v in recap.items()}
enregistrer(f"3b : + filtrage guide [{STATUT}]", res_test_3b, res_date_3b)
tableau_recap()

,version,type,précision,rappel,F1,support
0,"Baseline 1 : spaCy seul [test 1, utilisé pendant le développement (optimiste)]",PER,0.75,0.43,0.55,7
1,"Baseline 1 : spaCy seul [test 1, utilisé pendant le développement (optimiste)]",ORG,0.47,0.69,0.56,13
2,"Baseline 1 : spaCy seul [test 1, utilisé pendant le développement (optimiste)]",LOC,0.29,0.50,0.36,4
3,"Baseline 1 : spaCy seul [test 1, utilisé pendant le développement (optimiste)]",TOUT (PER+ORG+LOC),0.47,0.58,0.52,24
4,"Baseline 1 : spaCy seul [test 1, utilisé pendant le développement (optimiste)]",DATE (à part),0.00,0.00,0.00,3
5,"3a : + règles EntityRuler [test 1, utilisé pendant le développement (optimiste)]",PER,0.83,0.71,0.77,7
6,"3a : + règles EntityRuler [test 1, utilisé pendant le développement (optimiste)]",ORG,0.72,1.00,0.84,13
7,"3a : + règles EntityRuler [test 1, utilisé pendant le développement (optimiste)]",LOC,0.44,1.00,0.62,4
8,"3a : + règles EntityRuler [test 1, utilisé pendant le développement (optimiste)]",TOUT (PER+ORG+LOC),0.67,0.92,0.77,24
9,"3a : + règles EntityRuler [test 1, utilisé pendant le développement (optimiste)]",DATE (à part),1.00,1.00,1.00,3


**Fin de l'étape 3b.** Prochaine étape (3c) : comparer `sm` / `md` / `lg` avec **les mêmes** règles et filtres.